In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve().parent))

# Phase 3 — Rules Engine + Isolation Forest (fraud intelligence, no final decision)

**Goal:** deterministic AML rules (`src/rules_engine.py`) + unsupervised Isolation Forest (`src/anomaly_detector.py`) as *independent, interpretable* signals.
Outputs: `artifacts/fraud_intelligence_features.csv`, `artifacts/rule_audit_log.json`, `artifacts/anomaly_detector.joblib`, `artifacts/isolation_forest_metadata.json`, `reports/rules_and_anomaly_report.md`.
> No final decision is made here — fusion happens in a later phase. Metrics focus on precision/recall/F1, never accuracy.

In [ ]:
import json
import logging
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
log = logging.getLogger('phase3')

BASE_DIR = Path('.').resolve().parent
ART = BASE_DIR / 'artifacts'
CFG = BASE_DIR / 'config'
REP = BASE_DIR / 'reports'
ART.mkdir(exist_ok=True)
REP.mkdir(exist_ok=True)

from src import config as app_config
C = app_config.COLS
LABEL = C.get('label', 'is_suspicious')
TXID = C.get('transaction_id', 'transaction_id')

def load_rules_yaml(path: Path) -> dict:
    """Minimal parser for flat config/rules.yaml (avoids pyyaml dependency)."""
    cfg: dict = {}
    try:
        for line in path.read_text().splitlines():
            line = line.strip()
            if not line or line.startswith('#') or ':' not in line:
                continue
            k, v = line.split(':', 1)
            k, v = k.strip(), v.strip()
            if k in ('scores', 'severity_weights', 'anomaly_severity'):
                continue  # parsed below for anomaly_severity
            try:
                cfg[k] = json.loads(v) if v.startswith('[') else float(v) if '.' in v else int(v)
            except Exception:
                cfg[k] = v
    except Exception as e:
        log.warning('rules.yaml parse failed: %s', e)
    # anomaly_severity: {LOW: 0.3, MEDIUM: 0.6, HIGH: 0.8}
    bands = {'LOW': 0.3, 'MEDIUM': 0.6, 'HIGH': 0.8}
    try:
        text = path.read_text()
        import re
        m = re.search(r'anomaly_severity:\s*\{([^}]+)\}', text)
        if m:
            for part in m.group(1).split(','):
                kk, vv = part.split(':')
                bands[kk.strip()] = float(vv.strip())
    except Exception as e:
        log.warning('anomaly_severity parse failed, defaults %s: %s', bands, e)
    cfg['anomaly_severity'] = bands
    return cfg

RULES_CFG = load_rules_yaml(CFG / 'rules.yaml')
SEV_BANDS = RULES_CFG['anomaly_severity']
log.info('anomaly_severity bands: %s', SEV_BANDS)

try:
    IF_FEATURES = json.loads((ART / 'isolation_forest_features.json').read_text())
    log.info('IF features (%d): %s', len(IF_FEATURES), IF_FEATURES)
except Exception as e:
    log.warning('isolation_forest_features.json missing: %s', e)
    IF_FEATURES = []

CONTAMINATION = app_config.ISOLATION_FOREST.get('contamination', 0.10)
log.info('BASE=%s ART=%s contamination=%s', BASE_DIR, ART, CONTAMINATION)

## PART A — Deterministic rules engine (`apply_rules` + `build_audit_log`)

Loads `artifacts/transaction_features.csv` (fallback: rebuild via `feature_store` on `train.csv`), checks required features (missing → warn + engine defaults), applies ~18 AML flags, outputs `rule_score`, `rule_score_weighted`, `rule_score_norm` (0–1 capped), `rules_triggered_count`, `max_rule_severity`, `rule_reasons`.

In [ ]:
from src.rules_engine import apply_rules, build_audit_log

FE_CSV = ART / 'transaction_features.csv'
df = None
try:
    if FE_CSV.exists():
        df = pd.read_csv(FE_CSV)
        log.info('loaded %s shape=%s', FE_CSV, df.shape)
    else:
        raise FileNotFoundError(str(FE_CSV))
except Exception as e:
    log.warning('transaction_features.csv unavailable (%s); fallback: build via feature_store on train.csv', e)
    try:
        train_csv = ART / 'train.csv'
        if not train_csv.exists():
            train_csv = BASE_DIR / 'labeled_transactions.csv'
        raw = pd.read_csv(train_csv)
        log.info('fallback raw %s shape=%s', train_csv, raw.shape)
        from src.feature_engineering import add_features
        from src.feature_store import add_phase2_features
        raw[C['timestamp']] = pd.to_datetime(raw[C['timestamp']])
        tmp = add_features(raw)
        df = add_phase2_features(tmp)
        df.to_csv(FE_CSV, index=False)
        log.info('fallback features built shape=%s saved to %s', df.shape, FE_CSV)
    except Exception as e2:
        log.exception('fallback feature build failed: %s', e2)
        raise

# Required-feature check (engine fills defaults via _col, so missing -> warn + continue)
REQUIRED = ['amount_ratio', 'txns_last_day', 'txns_last_7d', 'new_beneficiary',
            'days_since_prev', 'fan_in_7d', 'fan_out_7d', 'passthrough_ratio',
            'burst_score', 'is_weekend']
missing = [c for c in REQUIRED if c not in df.columns]
if missing:
    log.warning('missing rule input features (engine will default to 0): %s', missing)
else:
    log.info('all required rule features present')
for must in [TXID, C.get('account_id', 'account_id'), C.get('timestamp', 'date')]:
    if must not in df.columns:
        log.warning('key column missing: %s', must)

try:
    ruled = apply_rules(df)
    log.info('apply_rules ok shape=%s flagged=%d', ruled.shape, int((ruled['rule_score'] > 0).sum()))
except Exception:
    log.exception('apply_rules failed')
    raise

# Distribution + severity counts
try:
    print('--- rule_score distribution ---')
    print(ruled['rule_score'].describe().to_string())
    print('\ntriggered-count value_counts:')
    print(ruled['rules_triggered_count'].value_counts().sort_index().to_string())
    print('\nmax_rule_severity counts:')
    print(ruled['max_rule_severity'].value_counts().to_string())
    print('\nrule_score_weighted describe:')
    print(ruled['rule_score_weighted'].describe().to_string())
    print('\nrule_score_norm describe (0-1 capped):')
    print(ruled['rule_score_norm'].describe().to_string())
except Exception as e:
    log.warning('summary print failed: %s', e)

# Save fraud-intelligence columns
FI_COLS = [TXID, 'rule_score', 'rule_score_weighted', 'rule_score_norm',
           'rules_triggered_count', 'max_rule_severity']
try:
    fi = ruled[[c for c in FI_COLS if c in ruled.columns]].copy()
    fi.to_csv(ART / 'fraud_intelligence_features.csv', index=False)
    log.info('saved fraud_intelligence_features.csv shape=%s', fi.shape)
    print(fi.head(5).to_string())
except Exception:
    log.exception('saving fraud_intelligence_features.csv failed')
    raise

# Audit log (cap 200k records)
try:
    records = build_audit_log(ruled)
    log.info('audit records raw=%d; capping at 200k', len(records))
    records = records[:200000]
    with open(ART / 'rule_audit_log.json', 'w') as f:
        json.dump(records, f, indent=1)
    log.info('saved rule_audit_log.json records=%d', len(records))
except Exception:
    log.exception('audit log build/save failed')
    raise

## PART B — Isolation Forest on normal-only transactions

Validates `isolation_forest_features.json` (drops missing / constant / inf columns), trains `AnomalyDetector` on `is_suspicious == 0` only, saves model + metadata, scores all rows, and bands `anomaly_score` → severity via `config/rules.yaml anomaly_severity`.

In [ ]:
from src.anomaly_detector import AnomalyDetector

df = ruled.copy()
log.info('scoring base n=%d', len(df))

# Validate IF features: drop missing + constant + inf
kept, dropped = [], {}
for c in IF_FEATURES:
    if c not in df.columns:
        dropped[c] = 'missing'; continue
    s = df[c]
    if s.nunique(dropna=True) <= 1:
        dropped[c] = 'constant'; continue
    arr = pd.to_numeric(s, errors='coerce')
    if np.isinf(arr.replace([np.inf, -np.inf], np.nan).dropna()).any() or np.isinf(arr.fillna(0).values).any():
        dropped[c] = 'inf'; continue
    kept.append(c)
if dropped:
    log.warning('dropped IF features: %s', dropped)
if not kept:
    raise ValueError(f'all IF features invalid; dropped={dropped}')
log.info('IF kept (%d): %s', len(kept), kept)

X = df[kept].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)

labels_available = LABEL in df.columns
if labels_available:
    y_all = pd.to_numeric(df[LABEL], errors='coerce').fillna(0).astype(int)
    X_train = X[y_all == 0]
    log.info('labels available: normal=%d suspicious=%d', int((y_all == 0).sum()), int((y_all == 1).sum()))
    if len(X_train) == 0:
        log.warning('no normal samples; training on all data')
        X_train, y_train = X, None
    else:
        y_train = y_all[y_all == 0]
else:
    log.warning('label column %s missing; unsupervised fit on all rows', LABEL)
    y_all, y_train, X_train = None, None, X

try:
    det = AnomalyDetector()
    det.fit(X_train, y_train)
    det.save(ART / 'anomaly_detector.joblib')
    log.info('saved anomaly_detector.joblib')
except Exception:
    log.exception('AnomalyDetector fit/save failed')
    raise

meta = {'date': str(date.today()), 'n_train': int(len(X_train)), 'features': kept,
        'contamination': float(CONTAMINATION), 'labels_available': bool(labels_available)}
try:
    (ART / 'isolation_forest_metadata.json').write_text(json.dumps(meta, indent=1))
    log.info('saved isolation_forest_metadata.json: %s', meta)
except Exception:
    log.exception('metadata save failed')
    raise

# Score all splits (all rows in scope)
try:
    scored = det.score(X)
    df['anomaly_score'] = scored['anomaly_score'].clip(0, 1).values
    df['anomaly_flag'] = scored['anomaly_flag'].astype(int).values
    lo, med, hi = SEV_BANDS.get('LOW', 0.3), SEV_BANDS.get('MEDIUM', 0.6), SEV_BANDS.get('HIGH', 0.8)
    df['anomaly_severity'] = pd.cut(df['anomaly_score'], bins=[-0.01, lo, med, hi, 1.01],
                                     labels=['NORMAL', 'LOW', 'MEDIUM', 'HIGH'])
    log.info('scored all n=%d flagged=%d', len(df), int(df['anomaly_flag'].sum()))
    print(df[['anomaly_score', 'anomaly_flag', 'anomaly_severity']].describe(include='all').to_string())
    print('\nanomaly_severity counts:')
    print(df['anomaly_severity'].value_counts().to_string())
except Exception:
    log.exception('scoring failed')
    raise

## PART C — Combined interpretation + per-system evaluation (no final decision)

Scatter `rule_score_norm` vs `anomaly_score` quadrants are *described only*. Each system is evaluated independently vs labels (`rules: weighted >= 3`, `IF: flag`) with precision/recall/F1 — accuracy is reported for context only and never optimized. FP examples illustrate blind spots.

In [ ]:
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

# Combined scatter (interpretation only — NO final decision / NO fusion)
try:
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(df['rule_score_norm'], df['anomaly_score'], s=6, alpha=0.4)
    ax.set_xlabel('rule_score_norm (0-1 capped)')
    ax.set_ylabel('anomaly_score (0-1)')
    ax.set_title('Rules vs Anomaly scores (interpretation only)')
    ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
    fig.tight_layout()
    plt.show()
    print('Quadrants: low-rule/low-anomaly=likely normal; high-rule/low-anomaly=known typology;'
          ' low-rule/high-anomaly=novel outlier worth review; high/high=both systems agree (still NOT a decision).')
except Exception as e:
    log.warning('scatter failed: %s', e)

# Per-system confusion matrices vs labels
eval_lines = []
try:
    if y_all is None:
        msg = 'labels unavailable; skipping supervised evaluation.'
        log.warning(msg); eval_lines.append(msg)
        rules_pred = (df['rule_score_weighted'] >= 3).astype(int)
        if_pred = df['anomaly_flag'].astype(int)
    else:
        rules_pred = (df['rule_score_weighted'] >= 3).astype(int)
        if_pred = df['anomaly_flag'].astype(int)
        for name, pred in [('Rules (weighted>=3)', rules_pred), ('IsolationForest (flag)', if_pred)]:
            cm = confusion_matrix(y_all, pred, labels=[0, 1])
            p, r, f, _ = precision_recall_fscore_support(y_all, pred, labels=[0, 1], zero_division=0)
            acc = float((y_all.values == pred.values).mean())
            block = (f'{name}: cm=[[TN={cm[0,0]}, FP={cm[0,1]}], [FN={cm[1,0]}, TP={cm[1,1]}]] '
                     f'precision1={p[1]:.3f} recall1={r[1]:.3f} F1_1={f[1]:.3f} (accuracy={acc:.3f} for context only, NOT optimized)')
            print(block); eval_lines.append(block)
        print('Note: imbalanced labels — compare precision/recall/F1, never accuracy.')
except Exception:
    log.exception('evaluation failed')
    raise

# FP examples: 3 rule-FP + 3 IF-FP
fp_lines = []
SHOW = [TXID, 'rule_score', 'rule_score_weighted', 'rule_score_norm', 'max_rule_severity',
        'anomaly_score', 'anomaly_flag', 'anomaly_severity']
SHOW = [c for c in SHOW if c in df.columns]
try:
    if y_all is not None:
        rfp = df[(rules_pred == 1) & (y_all.values == 0)].head(3)
        afp = df[(if_pred == 1) & (y_all.values == 0)].head(3)
        print('--- 3 rule-FP rows (rules fire, label=0) ---')
        print(rfp[SHOW].to_string() if len(rfp) else 'none')
        print('--- 3 IF-FP rows (IF flags, label=0) ---')
        print(afp[SHOW].to_string() if len(afp) else 'none')
        fp_lines = ['rule-FP: ' + (rfp[SHOW].to_string() if len(rfp) else 'none'),
                      'IF-FP: ' + (afp[SHOW].to_string() if len(afp) else 'none')]
    else:
        fp_lines = ['labels unavailable; FP sampling skipped.']
except Exception as e:
    log.warning('FP sampling failed: %s', e)

# Markdown report
try:
    sev_ct = df['max_rule_severity'].value_counts().to_dict() if 'max_rule_severity' in df.columns else {}
    ase_ct = df['anomaly_severity'].value_counts().to_dict() if 'anomaly_severity' in df.columns else {}
    rep = ['# Rules + Anomaly Report (Phase 3)', '', f'Date: {date.today()}', '',
           f'n={len(df)} IF_features={kept} contamination={CONTAMINATION} labels_available={bool(y_all is not None)}', '',
           '## Rules', f'severity_counts={sev_ct}',
           f"flagged(rule_score>0)={int((df['rule_score'] > 0).sum())} weighted>=3={int((df['rule_score_weighted'] >= 3).sum())}", '',
           '## IsolationForest', f'anomaly_severity_counts={ase_ct}', f"flagged={int(df['anomaly_flag'].sum())}", '',
           '## Per-system evaluation (precision/recall/F1; accuracy context only)'] + eval_lines + ['',
           '## Quadrants (description only, NO decision)',
           'low/low=likely normal; high-rule/low-anomaly=known typology; low-rule/high-anomaly=novel outlier; high/high=agreement, still not a decision.', '',
           '## FP examples', ''] + fp_lines
    (REP / 'rules_and_anomaly_report.md').write_text('\n'.join(rep))
    log.info('saved reports/rules_and_anomaly_report.md')
except Exception:
    log.exception('report write failed')
    raise

print('PHASE 3 COMPLETE: rules + IF signals ready | NEXT PHASE: 04_financial_graph_construction.ipynb')